In [1]:
from netgen.meshing import Mesh
from ngsolve import *
from ngsolve.krylovspace import CGSolver
from netgen.occ import *
from ngsolve.webgui import Draw

import matplotlib.pylab as plt
import scipy.sparse as sp

In [2]:
def Capacitor3DGeometry():
    air_box = Box((-10, -10, -10), (10, 10, 10))
    air_box.faces.name = "Outer"  

    dielectric = Box((-2.5, -0.75, -2), (2.5, 0.75, 2))
    dielectric.name = "dielectric"

    electrode_positive = Box((-2.5, 0.75, -2), (2.5, 1.25, 2))
    electrode_positive.faces.name = "electrode_positive"

    electrode_negative = Box((-2.5, -1.25, -2), (2.5, -0.75, 2))
    electrode_negative.faces.name = "electrode_negative"

    air = air_box - dielectric
    air.name = "air"

    shape = Glue([air, dielectric])
    shape = shape - electrode_positive - electrode_negative

    return shape


def Capacitor3DMesh(shape, h_max):
    
    mesh = Mesh(OCCGeometry(shape, dim=3).GenerateMesh(maxh=h_max))

    return mesh


def Capacitor3DWeakForm(mesh, FE_order, epsr):

    fes = H1(mesh, order=FE_order, dirichlet="el.*")

    u = fes.TrialFunction()
    v = fes.TestFunction()

    a = BilinearForm(epsr*grad(u)*grad(v)*dx)
    precond = preconditioners.Local(a)

    return a, fes, precond


def Capacitor3DAssemble(a):
     
    pre = preconditioners.Local(a)
     
    with TaskManager():
        a.Assemble()

    return a
    


def Capacitor3DSolver(mesh, fes, a, precond):

    solution_gf = GridFunction(fes)
    solution_gf.Interpolate(mesh.BoundaryCF({"electrode_positive":1, "electrode_negative":-1 }), mesh.Boundaries(".*"))
        

    with TaskManager():
        inv = CGSolver(
            mat=a.mat,
            pre=precond,
            printrates='\r',
            maxiter=10000
        )
      
        solution_gf.vec.data -= inv*(a.mat * solution_gf.vec)

    return solution_gf



In [3]:
geo = Capacitor3DGeometry()

In [4]:
h_max = 0.2
mesh = Capacitor3DMesh(geo, h_max)

In [5]:
epsr_air = 1.0
epsr_dielectric = 4.0

epsr = mesh.MaterialCF({"air": epsr_air, "dielectric": epsr_dielectric})

In [ ]:
clipping = {"function": True, "pnt": (0, 0, 0), "vec": (0, 0, -1)}
Draw(epsr, mesh, draw_surf=False, draw_vol=True, clipping=clipping)

In [ ]:
FE_order = 2

a, fes, precond = Capacitor3DWeakForm(mesh, FE_order, epsr)

In [ ]:
a = Capacitor3DAssemble(a)

In [ ]:
phi_gf = Capacitor3DSolver(mesh, fes, a, precond)

In [ ]:
Draw (phi_gf, deformation=True, scale=5, clipping=clipping);

In [ ]:
fes_flux = HCurl(mesh, order=FE_order-1)

E_gf = GridFunction(fes_flux)
E_gf.Set(-grad(phi_gf))

In [ ]:
N = 100
p = [(-10 + 0.2*i, -10 + 0.2*j, -10 + 0.2*k) for i in range(N) for j in range(N) for k in range(N)] 

fieldlines = E_gf._BuildFieldLines(mesh, p, num_fieldlines=400, length=3)

Draw(E_gf, mesh,  "X", draw_vol=True, draw_surf=True, objects=[fieldlines], \
     autoscale=True, min = 0, max = 1, settings={"Objects": {"Surface": False}});